In [5]:
import numpy as np
import pandas as pd

age0_path = 'data_raw/ICPSR_37871/DS0001/37871-0001-Data.dta'
age4_path = 'data_raw/ICPSR_37871/DS0009/37871-0009-Data.dta'

In [7]:
age0 = pd.read_stata(age0_path, convert_categoricals=False)
age4 = pd.read_stata(age4_path, convert_categoricals=False)

In [9]:
age0.shape, age4.shape

((1050, 626), (1000, 1123))

In [11]:
# Retain 1,000 mothers in study sample and sort by their ID 

age0 = age0.query('STUDYSAMPLEA0 == 1').sort_values(by='PUBLICSAMPLEID')
age4 = age4.sort_values(by='PUBLICSAMPLEID')

len(age0), (age0['PUBLICSAMPLEID'].to_numpy() == age4['PUBLICSAMPLEID'].to_numpy()).sum()

(1000, np.int64(1000))

In [17]:
# age4 variables including any relevant covariates

age4_variables = {
    'treat': 'TREAT', 
    'site': 'SITE', 
    'vocab': 'CROWPVTA4',
    'ef': 'CMEFSSTANDARDA4', 
    'cbcl': 'CCBCLINDEXA4',
    'eeg': 'CHIGHFREQCOMPA4A4',
    'eeg_final': 'CEEGFINALSAMPLEA4A4',
    'matrices': 'CMATRICESRAWA4',
    'reading': 'CREADINGRAWA4',
    'diagnosis': 'CCDEVTDIAGA4',
    'assessor': 'ASSESSINTERVIEWERA4', 
    'spanish': 'ANYASSESSSPANISHA4',
    'under4': 'ASSESSUNDER4YOA4', 
}

# baseline (age0) variables including any relevant covariates
age0_variables = {
    'female': 'CFEMALEA0', 
    'birthweight_oz': 'CWEIGHTOZA0', 
    'gest_age': 'CGESTAGEWKSA0', 
    'mom_age': 'MCALCAGEA0', 
    'mom_educ': 'MEDLEVELA0', 
    'mom_race': 'MRACEA0', 
    'marital': 'MRELATESTATUSA0', 
    'mom_health': 'MHEALTHA0', 
    'cesd': 'MDEPRESSIONA0', 
    'smoked': 'MCIGDURINGA0', 
    'alcohol': 'MALCDURINGA0', 
    'n_children': 'HHNCHILDA0', 
    'n_adults': 'HHNADULTA0', 
    'biodad_in_hh': 'HHBIODADA0', 
    'hh_income': 'HHCOMBINEDINCOMEA0', 
    'net_worth': 'HHNETWORTHA0', 
}

len(age4_variables), len(age0_variables)

(13, 16)

In [25]:
# Create one data frame to store both files and replace missing codes (-777, -888, -999) with NaN

df = age4[['PUBLICSAMPLEID']].copy()

for new, old in age4_variables.items():
    x = age4[old].to_numpy(dtype=float, copy=True)
    x[x == -777] = np.nan
    x[x == -888] = np.nan
    x[x == -999] = np.nan
    df[new] = x

for new, old in age0_variables.items():
    x = age0[old].to_numpy(dtype=float, copy=True)
    x[x == -777] = np.nan
    x[x == -888] = np.nan
    x[x == -999] = np.nan
    df[new] = x

df.shape, df.query('treat == 1').shape[0], df.query('treat == 0').shape[0]

((1000, 30), 400, 600)

In [27]:
# Replicate the paper's EEG sample 

eeg = df['eeg'].to_numpy(copy=True)
eeg_final = df['eeg_final'].to_numpy()
eeg[eeg_final == 0] = np.nan
df['eeg'] = eeg

df = df.drop(columns='eeg_final')

outcomes = ['vocab', 'ef', 'cbcl', 'eeg', 'matrices', 'reading', 'diagnosis']
df[outcomes].describe().loc['count']

vocab        700.0
ef           783.0
cbcl         882.0
eeg          637.0
matrices     769.0
reading      754.0
diagnosis    881.0
Name: count, dtype: float64

In [31]:
# low-cash group

df.query('treat == 0')[outcomes].describe().loc[['count', 'mean', 'std']]

,vocab,ef,cbcl,eeg,matrices,reading,diagnosis
count,404.000000,449.000000,513.000000,359.000000,443.000000,434.000000,511.000000
mean,40.997525,93.746102,22.175439,2.920130,7.625282,3.016129,0.136986
std,16.204417,10.480323,12.928153,1.369341,3.359257,2.242198,0.344170


In [29]:
# high-cash group 

df.query('treat == 1')[outcomes].describe().loc[['count', 'mean', 'std']]

,vocab,ef,cbcl,eeg,matrices,reading,diagnosis
count,296.000000,334.000000,369.000000,278.000000,326.000000,320.00000,370.000000
mean,39.199324,94.553892,22.392954,3.051135,7.303681,2.80625,0.145946
std,15.516775,10.595558,12.503915,1.349461,3.308235,2.17422,0.353530


In [37]:
# Standardize continuous outcomes by the low-cash group's SD

continuous_vars = ['vocab', 'ef', 'cbcl', 'eeg', 'matrices', 'reading']
low = df.query('treat == 0')

for c in continuous_vars: 
    df[c + "_z"] = (df[c] - low[c].mean()) / low[c].std()

z_names = [c + '_z' for c in continuous_vars]
df.query('treat == 1')[z_names].describe().loc[['count', 'mean', 'std']]

,vocab_z,ef_z,cbcl_z,eeg_z,matrices_z,reading_z
count,296.000000,334.000000,369.000000,278.000000,326.000000,320.000000
mean,-0.110970,0.077077,0.016825,0.095671,-0.095736,-0.093604
std,0.957565,1.010995,0.967185,0.985482,0.984811,0.969682


In [39]:
# Save cleaned table

df.to_csv('data_clean/bfy_age4_clean.csv', index=False)

df.shape

(1000, 35)